# 空気中の放電：1D と 2D（軸対称）の比較

**目的**：同じ空気・同じ輸送データで 1D と 2D の計算を行い、次元による違いを見る。

| | 1D（`standard_1d`） | 2D 軸対称（`standard_2d`） |
|---|---|---|
| 形状 | 無限に広い平面状の電離フロント | 中心軸上の細いチャネル（ストリーマー） |
| 元の設定 | `streamer_1d.cfg` | `streamer_cyl.cfg` |
| 気体・輸送データ | 空気（N2:O2 = 8:2, 1 bar）, `td_air_siglo_swarm.txt` | 同じ |
| 光電離 | なし | なし |

**予想**：空気の絶縁破壊電界はおよそ $E_k \approx 3$ MV/m。

* 1D の平面フロントでは、フロントの前方の電界は背景電界のまま強まらない。そのため $E < E_k$ では進めない。
* 2D のストリーマーでは、細い先端に空間電荷が集まって電界が強まる（電界集中）。そのため $E < E_k$ でも進める。

### 1D の設定で注意したこと
* **種は中性のまま**にした。1D で正イオンの種を置くと、無限に広い平面状の電荷層になり、
  初期電界が $10^8$ V/m を超えて計算が壊れる。
* **背景の電子密度は 0 のまま**にした。1D で $E > E_k$ かつ背景電子があると、計算領域全体が
  一斉になだれ増倍を起こして（フロントではなく一様な絶縁破壊）、`dt too small` で止まる。
  そのため 1D で進むのは、電子が流れていく向きの**負極性フロント**だけになる。
* 2D の例は**正極性ストリーマー**（背景電界と同じ向きに進む）なので、極性が違う点に注意。

## 0. 準備

In [ ]:
import matplotlib.pyplot as plt
import matplotlib_fontja  # 日本語フォント
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from afivo_helpers import (REPO, run_streamer, load_log, front_velocity,
                           lineout_settings, load_lineout)

L1D = 32e-3   # 1D の計算領域の長さ (m)
L2D = 16e-3   # 2D の計算領域の長さ (m)

# 軸方向の分布をテキストで出力する設定（1D は全体、2D は中心軸 r = 0 上）
LINE_1D = lineout_settings(ndim=1)
LINE_2D = lineout_settings(ndim=2)

## 1. 同じ背景電界（2 MV/m < $E_k$）で比べる

2D の例と同じ 2 MV/m で、1D と 2D を 8 ns まで計算します（2D は約 2〜3 分）。

In [ ]:
E0 = -2.0e6      # 背景電界 (V/m)、負は -z 向き
T_END = 8e-9

base_1d = run_streamer("standard_1d", "streamer_1d.cfg", "cmp_1d_E2.0",
                       {"field_amplitude": E0, "end_time": T_END, **LINE_1D}, threads=1)
base_2d = run_streamer("standard_2d", "streamer_cyl.cfg", "cmp_2d_E2.0",
                       {"field_amplitude": E0, "end_time": T_END, **LINE_2D}, threads=4,
                       show_every=8)
log1, log2 = load_log(base_1d), load_log(base_2d)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6), constrained_layout=True)
for log, lbl in [(log1, "1D"), (log2, "2D 軸対称")]:
    t = log["time"] * 1e9
    ax[0].semilogy(t, log["max(n_e)"], label=lbl)
    ax[1].plot(t, log["max(E)"] / 1e6, label=lbl)
ax[0].set_ylabel("最大電子密度 (m$^{-3}$)")
ax[1].set_ylabel("最大電界 (MV/m)")
ax[1].axhline(3.0, color="gray", ls="--", lw=1)
ax[1].text(0.2, 3.3, "$E_k$ ≈ 3 MV/m", color="gray")

t2 = log2["time"][1:] * 1e9
ax[2].plot(t2, log2["ne_zmin"][1:] * 1e3, color="C1", label="2D 先端位置")
ax[2].set_ylabel("先端位置 z (mm)")
for a in ax:
    a.set_xlabel("t (ns)"); a.grid(alpha=0.3); a.legend()
fig.suptitle(f"背景電界 {abs(E0)/1e6:.1f} MV/m")

v2d = front_velocity(log2["time"], log2["ne_zmin"], 1e-3, L2D - 1e-3)
print(f"2D ストリーマーの速度: {v2d/1e3:.0f} km/s,  先端の最大電界: {log2['max(E)'].iloc[-1]/1e6:.1f} MV/m")
print(f"1D: 最大電子密度 {log1['max(n_e)'].iloc[0]:.2e} → {log1['max(n_e)'].iloc[-1]:.2e} m^-3（付着で減少）")

## 2. 軸方向の分布を比べる

最後の時刻（8 ns）の電子密度と電界を、z 方向に沿って比べます。
2D は中心軸（r = 0）上の値です。比較のため、1D は $E > E_k$ でフロントができる 8 MV/m の計算も並べます。

In [ ]:
base_1d_8 = run_streamer("standard_1d", "streamer_1d.cfg", "cmp_1d_E8.0",
                         {"field_amplitude": -8e6, "end_time": T_END, **LINE_1D}, threads=1)

fig, ax = plt.subplots(2, 1, figsize=(9, 6), sharex=True, constrained_layout=True)
for base, lbl, c in [(base_2d, "2D 軸上, 2 MV/m", "C1"),
                     (base_1d, "1D, 2 MV/m", "C0"),
                     (base_1d_8, "1D, 8 MV/m", "C2")]:
    df, t = load_lineout(base)            # 最後の出力
    z = df["y"] if "y" in df else df["x"]   # 2D は y が軸方向
    ne, E = df["e"], df["electric_fld"]
    ax[0].semilogy(z * 1e3, np.maximum(ne, 1e10), label=lbl, color=c)
    ax[1].plot(z * 1e3, E / 1e6, label=lbl, color=c)
ax[0].set_ylabel("電子密度 (m$^{-3}$)"); ax[0].set_ylim(1e12, 1e21)
ax[1].set_ylabel("電界 (MV/m)"); ax[1].set_xlabel("z (mm)")
ax[1].axhline(3.0, color="gray", ls="--", lw=1)
for a in ax:
    a.grid(alpha=0.3); a.legend(loc="upper right")
fig.suptitle(f"t = {t*1e9:.1f} ns の軸方向分布")

## 3. 1D 平面フロントの速度と電界の関係

1D で背景電界を変えて、負極性フロントの速度 $v(E)$ を求めます。
2D ストリーマーの点は、**先端の最大電界**に対する速度として同じ図に置きます
（先端の局所電界が、平面フロントの何 MV/m に相当するかの目安になります）。

1D の計算は1スレッドで軽いので、5本を並列に走らせます（強い電界ほど時間がかかり、17 MV/m は約3分）。

In [ ]:
fields_1d = [6e6, 8e6, 10e6, 13e6, 17e6]   # V/m

def run_1d(E):
    name = f"cmp_1d_scan_E{E/1e6:.0f}"
    b = run_streamer("standard_1d", "streamer_1d.cfg", name,
                     {"field_amplitude": -E, "end_time": 10e-9},
                     threads=1, quiet=True)
    return E, load_log(b)

with ThreadPoolExecutor(max_workers=len(fields_1d)) as ex:
    scan = dict(ex.map(run_1d, fields_1d))

rows = []
for E, log in scan.items():
    v = front_velocity(log["time"], log["ne_zmax"], 1e-3, L1D - 1e-3)
    rows.append({"E (MV/m)": E / 1e6, "v (km/s)": v / 1e3,
                 "max n_e (m^-3)": log["max(n_e)"].max()})
table = pd.DataFrame(rows)
table

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for E, log in scan.items():
    ok = (log["ne_zmax"] > 0) & (log["ne_zmax"] < L1D)
    ax[0].plot(log["time"][ok] * 1e9, log["ne_zmax"][ok] * 1e3, label=f"{E/1e6:.0f} MV/m")
ax[0].set_xlabel("t (ns)"); ax[0].set_ylabel("1D フロント位置 z (mm)")
ax[0].legend(); ax[0].grid(alpha=0.3)

ax[1].plot(table["E (MV/m)"], table["v (km/s)"], "o-", label="1D 平面フロント（負極性）")
E_tip = log2["max(E)"].iloc[len(log2) // 2:].mean() / 1e6
ax[1].plot([E_tip], [v2d / 1e3], "s", ms=10, color="C1",
           label=f"2D ストリーマー（正極性）\n背景 {abs(E0)/1e6:.0f} MV/m, 先端 {E_tip:.0f} MV/m")
ax[1].axvline(abs(E0) / 1e6, color="C1", ls=":", lw=1)
ax[1].axvline(3.0, color="gray", ls="--", lw=1)
ax[1].set_xlabel("電界 (MV/m)"); ax[1].set_ylabel("速度 (km/s)")
ax[1].set_xlim(0, None); ax[1].set_ylim(0, None)
ax[1].legend(); ax[1].grid(alpha=0.3)

## 4. 考察のポイント

* **同じ 2 MV/m でも結果が正反対**：1D は付着で電子が減るだけ、2D は先端の電界集中で進む。
  2D の先端電界は背景の何倍になっているか？
* **1D の $v(E)$**：$E_k$ を超えると速度が電界とともに増える。
  2D ストリーマーの点は 1D の曲線の上下どちらにあるか？
  （極性の違い・背景電離の有無・先端の曲率などが原因の候補）
* **注意**：1D の 13・17 MV/m ではフロントが 10 ns の間ずっと加速していて、速度が一定になっていない。
  表の速度は後半 5 ns の平均で、定常速度ではない。定常速度を求めるには、計算時間や計算領域を延ばす必要がある。
  また、6 MV/m のフロント位置が階段状なのは、格子の幅で位置が刻まれるため。
* **次の比較の候補**：
  * 2D で背景電界を変える（`run_streamer.ipynb` の 5 節）
  * 2D で負極性ストリーマーを作り、1D と極性を揃える
  * 光電離（`[photoi] enabled = t`）を入れて、背景電離の代わりにする